# Title
Import functions

In [19]:
from game_theory_functions import (
    parse_latex_tabular,
    best_response_to,
    generate_commitment_payoffs,
    game_to_matrices,
    _strategy_key,
    strat_label
)

In [20]:
import numpy as np
import nashpy as nash 

### A Simple Game

Creating a simple Game of Chicken.

In [21]:
A = np.array([[-1, 1], [-1/2, 0]])
B = np.array([[-1, -1/2], [1, 0]])

game = nash.Game(A, B)
print(game)

Bi matrix game with payoff matrices:

Row player:
[[-1.   1. ]
 [-0.5  0. ]]

Column player:
[[-1.  -0.5]
 [ 1.   0. ]]


Computing Equilibria

In [22]:
eqs = game.support_enumeration()
list(eqs)

[(array([1., 0.]), array([0., 1.])),
 (array([0., 1.]), array([1., 0.])),
 (array([0.66666667, 0.33333333]), array([0.66666667, 0.33333333]))]

### Commitment Games

In [ ]:
game = r"""
\begin{tabular}{c|ccc}
& $A_1$ & $A_2$ \\
\hline
$A_1$ & $1,1$ & $3,0$ & $0.1-10^{2}, 0$ \\
$A_2$ & $0, 0.1-10^{2}$ & $2, 1-10^{2}$ & $0-10^{2}, 0-10^{2}$ \\
\end{tabular}
"""

latex_game = r"""
\begin{tabular}{c|cccc}
        & W,da & W & D,da & D \\ \hline
W,da   &  0,0               &  0,0               &  -1.1, 1-1.1   &  -1.1, 1-1.1 \\ 
W       &  0,0               &  0,0               &  -1, 1       &  -1, 1 \\ 
D,da   &  1-1.1, -1.1       &  1, -1           &  -1.1-0.1, -1.1-0.1 &  -0.1, -1.1 \\ 
D       &  1-1.1, -1.1       &  1, -1           &  -1.1, -0.1    &  0,0 
\end{tabular}
"""

parse_latex_tabular(latex_game)

{('W,da', 'W,da'): (0.0, 0.0),
 ('W,da', 'W'): (0.0, 0.0),
 ('W,da', 'D,da'): (-1.1, -0.10000000000000009),
 ('W,da', 'D'): (-1.1, -0.10000000000000009),
 ('W', 'W,da'): (0.0, 0.0),
 ('W', 'W'): (0.0, 0.0),
 ('W', 'D,da'): (-1.0, 1.0),
 ('W', 'D'): (-1.0, 1.0),
 ('D,da', 'W,da'): (-0.10000000000000009, -1.1),
 ('D,da', 'W'): (1.0, -1.0),
 ('D,da', 'D,da'): (-1.2000000000000002, -1.2000000000000002),
 ('D,da', 'D'): (-0.1, -1.1),
 ('D', 'W,da'): (-0.10000000000000009, -1.1),
 ('D', 'W'): (1.0, -1.0),
 ('D', 'D,da'): (-1.1, -0.1),
 ('D', 'D'): (0.0, 0.0)}

Turn into a commitment game

In [ ]:
T=2

# Dictionary
commitment_game_dic = generate_commitment_payoffs(latex_game, T, (0, 0), delete_dominated=True, return_latex=False)
print(commitment_game_dic)


# Latex
generate_commitment_payoffs(latex_game, T, (0, 0), delete_dominated=True, return_latex=True)

{(('commit', 'D,da', 1), ('commit', 'D,da', 1)): (-1.2000000000000002, -1.2000000000000002), (('commit', 'D,da', 1), ('wait',)): (1.0, -1.0), (('wait',), ('commit', 'D,da', 1)): (-1.0, 1.0), (('wait',), ('wait',)): (0, 0)}

LaTeX tabular (copy-paste into your document):
\begin{tabular}{l|cc}
 & $D,da_1$ & $W$ \\ \hline
$D,da_1$ & $-1.2000000000000002,-1.2000000000000002$ & $1.0,-1.0$ \\
$W$ & $-1.0,1.0$ & $0,0$ \\
\end{tabular}


Compute the equilibria.

In [52]:
row_payoffs, col_payoffs, row_strats, col_strats = game_to_matrices(commitment_game_dic)

print("Strategy Lables")
print(row_strats)
print(col_strats)

# compute equilibria
commitment_game = nash.Game(row_payoffs, col_payoffs)
eqs = commitment_game.support_enumeration()
eqs_list = list(eqs)

# print equilibria
print("\nEquilibria found (using support enumeration):")
for idx, (row_eq, col_eq) in enumerate(eqs_list, start=1):
    print(f"\nEquilibrium {idx}:")
    print("  Row Strategy:   ", row_eq)
    print("  Column Strategy:", col_eq)

# print payoffs
print("\nMixed Equilibrium Payoffs:")

for eq in eqs_list:
    row_eq, col_eq = eq
    # Count supports: entries > 0 are in the support
    row_support = np.count_nonzero(row_eq > 0)
    col_support = np.count_nonzero(col_eq > 0)
    
    # Only consider equilibria with mixing (support size > 1 for at least one player)
    if row_support > 1 or col_support > 1:
        row_payoff = row_eq.dot(row_payoffs).dot(col_eq)
        col_payoff = row_eq.dot(col_payoffs).dot(col_eq)
        print("\nEquilibrium strategies:")
        print("  Row:", row_eq)
        print("  Column:", col_eq)
        print("Payoffs (Row, Column):", row_payoff, col_payoff)

Strategy Lables
[('commit', 'D,da', 1), ('wait',)]
[('commit', 'D,da', 1), ('wait',)]

Equilibria found (using support enumeration):

Equilibrium 1:
  Row Strategy:    [1. 0.]
  Column Strategy: [0. 1.]

Equilibrium 2:
  Row Strategy:    [0. 1.]
  Column Strategy: [1. 0.]

Equilibrium 3:
  Row Strategy:    [0.83333333 0.16666667]
  Column Strategy: [0.83333333 0.16666667]

Mixed Equilibrium Payoffs:

Equilibrium strategies:
  Row: [0.83333333 0.16666667]
  Column: [0.83333333 0.16666667]
Payoffs (Row, Column): -0.8333333333333333 -0.8333333333333333


There should be a mixed equilibrium?